In [16]:
import pandas as pd
import numpy as np
import tqdm

In [18]:
!pip install python-calamine

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 858.6/858.6 kB 3.2 MB/s eta 0:00:0000:0100:01


In [20]:
df_wide = pd.read_excel('pswci2_05_wide.xlsx', engine='calamine')
df_wide.head()

,pid,p05,nonresponse05,workperiod14,accident,injurytype,injurypart,con16,acc1,hire1,...,I01009001,I01009002,I01009003,I01010001,I01010002,I01010003,I01011001,I01011001t,I01012001,I01012001t
0,1,2,2.0,7,1,1,9,6,2014,2013,...,2,NaN,NaN,1,NaN,2.0,1,NaN,3,NaN
1,2,1,NaN,10,2,13,1,6,2012,2008,...,2,NaN,NaN,2,NaN,NaN,5,NaN,6,NaN
2,3,1,NaN,7,1,1,9,6,2014,2013,...,2,NaN,NaN,2,NaN,NaN,3,NaN,2,NaN
3,4,1,NaN,12,2,13,1,6,2015,2009,...,2,NaN,NaN,1,3000.0,NaN,7,NaN,1,NaN
4,5,1,NaN,8,2,13,1,6,2012,2010,...,2,NaN,NaN,2,NaN,NaN,1,NaN,1,NaN


In [22]:
df_main = pd.read_excel('pswci2_05_main.xlsx', engine='calamine')
df_main.head()

,pid,p05,nonresponse03,workperiod14,accident,injurytype,injurypart,con16,acc1,hire1,...,I05009002,I05009003,I05010001,I05010002,I05010003,I05011001,I05011001t,I05012001,I05012001t,filter_$
0,1,2,2.0,7,1,1,9,6,2014,2013,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,NaN,10,2,13,1,6,2012,2008,...,NaN,NaN,2.0,NaN,NaN,5.0,NaN,6.0,NaN,0.0
2,3,1,NaN,7,1,1,9,6,2014,2013,...,9000.0,NaN,2.0,NaN,NaN,4.0,NaN,1.0,NaN,0.0
3,4,1,NaN,12,2,13,1,6,2015,2009,...,NaN,NaN,1.0,10000.0,NaN,7.0,NaN,1.0,NaN,0.0
4,5,1,NaN,8,2,13,1,6,2012,2010,...,NaN,NaN,2.0,NaN,NaN,1.0,NaN,1.0,NaN,0.0


소비자물가지수, 서울시 생활임금

In [23]:
# 통계청 소비자물가지수 (2020=100 기준)
CPI = {3: 97.4, 4: 102.5, 5: 107.7}

# 서울시 생활임금 (월, 만원)
LIVING_WAGE = {3: 197.8, 4: 208.5, 5: 219.8}

ALPHA    = 0.4
NAN_VALS = [9999998, 9999999]

In [24]:
# 5차 실제 응답자 필터링(p05==1)
df = df_wide[df_wide['p05'] == 1].copy()
print(f"5차 응답자: {len(df)}명")

5차 응답자: 2728명


In [25]:
# ============================================================
# 소득 변수 정제
#    대상: 개인소득총계(H0x003036), 근로소득(H0x002005), 휴업급여(H0x003001)
#    처리: 결측코드 → NaN, 단위 만원/연 → 만원/월(÷12)
# ============================================================

income_raw_cols = (
    [f'H0{w}003036' for w in range(1, 6)] +   # 개인소득총계
    [f'H0{w}002005' for w in range(1, 6)] +   # 근로소득
    [f'H0{w}003001' for w in range(1, 6)]     # 휴업급여
)

for col in income_raw_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(NAN_VALS, np.nan)

# 월 단위 파생 변수 생성
for w in range(1, 6):
    for suffix in ['003036', '002005', '003001']:
        col_ann = f'H0{w}{suffix}'
        col_mon = f'H0{w}{suffix}_m'          # _m = monthly
        if col_ann in df.columns:
            df[col_mon] = df[col_ann] / 12

print("\n[차수별 개인소득총계 (월, 만원)]")
for w in range(1, 6):
    col = f'H0{w}003036_m'
    s = df[col].dropna()
    print(f"  {w}차: n={len(s)}, 평균={s.mean():.1f}, 중앙={s.median():.1f}")


[차수별 개인소득총계 (월, 만원)]
  1차: n=2728, 평균=336.5, 중앙=281.2
  2차: n=2654, 평균=204.9, 중앙=185.0
  3차: n=2650, 평균=202.6, 중앙=195.0
  4차: n=2693, 평균=211.5, 중앙=200.0
  5차: n=2728, 평균=207.0, 중앙=200.0


In [26]:
# ============================================================
# RIRI 계산 - 소득회복율
#    RIRI_w = (Q_irr_w × W_pre / LW_w) × (CPI_pre / CPI_w) × 100
#
#    Q_irr  = H0x002005_m / H0x003036_m   (근로소득 비중, 0~1 클리핑)
#    W_pre  = H03003036_m                 (3차 기준 월 소득)
#    LW_w   = 해당 차수 생활임금 (월, 만원)
#    post 차수: 3차(자기기준), 4차, 5차
# ============================================================

W_pre = df['H03003036_m']  # 3차 기준 월 소득

for w in [3, 4, 5]:
    total_m = df[f'H0{w}003036_m']
    earn_m  = df[f'H0{w}002005_m'].fillna(0)

    # Q_irr: 총소득 > 0인 경우만 계산
    q_irr = np.where(
        total_m.notna() & (total_m > 0),
        (earn_m / total_m).clip(0, 1),
        np.nan
    )

    riri = (q_irr * (W_pre / LIVING_WAGE[w])) * (CPI[3] / CPI[w]) * 100

    # 상위 1% 윈저라이징
    upper = np.nanpercentile(riri, 99)
    riri  = np.where(riri > upper, upper, riri)

    df[f'RIRI_w{w}'] = riri

    n_valid = (~np.isnan(riri)).sum()
    print(f"\nRIRI_w{w}: 유효={n_valid} | "
          f"평균={np.nanmean(riri):.1f} | 중앙={np.nanmedian(riri):.1f} | "
          f"≥100: {(riri>=100).sum()}명({np.nanmean(riri>=100)*100:.1f}%) | "
          f"<50: {(riri<50).sum()}명({np.nanmean(riri<50)*100:.1f}%)")



RIRI_w3: 유효=2464 | 평균=86.4 | 중앙=84.3 | ≥100: 1094명(40.1%) | <50: 859명(31.5%)

RIRI_w4: 유효=2490 | 평균=72.5 | 중앙=69.1 | ≥100: 825명(30.2%) | <50: 1049명(38.5%)

RIRI_w5: 유효=2501 | 평균=67.5 | 중앙=65.2 | ≥100: 764명(28.0%) | <50: 1067명(39.1%)


In [27]:
# ============================================================
# RIRI 3분류 레이블
#    2 = 완전회복 (RIRI ≥ 100)
#    1 = 부분회복 (50 ≤ RIRI < 100)
#    0 = 미회복   (RIRI < 50)
# ============================================================

def classify_riri(x):
    if pd.isna(x):   return np.nan
    elif x >= 100:   return 2
    elif x >= 50:    return 1
    else:            return 0

for w in [3, 4, 5]:
    df[f'RIRI_class_w{w}'] = df[f'RIRI_w{w}'].apply(classify_riri)
    dist  = df[f'RIRI_class_w{w}'].value_counts().sort_index()
    total = dist.sum()
    label = {0: '미회복', 1: '부분회복', 2: '완전회복'}
    print(f"{w}차: " + " | ".join(
        [f"{label[k]}={v}({v/total*100:.1f}%)" for k, v in dist.items()]))


3차: 미회복=859(34.9%) | 부분회복=511(20.7%) | 완전회복=1094(44.4%)
4차: 미회복=1049(42.1%) | 부분회복=616(24.7%) | 완전회복=825(33.1%)
5차: 미회복=1067(42.7%) | 부분회복=670(26.8%) | 완전회복=764(30.5%)


In [28]:
# ============================================================
# 장해등급 변수 처리
#    disa052  : 장해 유무 (1=있음, 2=없음)
#    disa0515 : 실제 등급 (1~14=유장해, 15=무장해)
#    → disa_class: 0=무장해, 1=중증(1~7급), 2=경증(8~14급)
# ============================================================

df['disa0515_num'] = pd.to_numeric(df['disa0515'], errors='coerce')

def classify_disability(g):
    if pd.isna(g) or g == 15:  return 0   # 무장해
    elif g <= 7:               return 1   # 중증
    else:                      return 2   # 경증

df['disa_class'] = df['disa0515_num'].apply(classify_disability)

dist_d = df['disa_class'].value_counts().sort_index()
print(f"\n장해등급: 무장해={dist_d.get(0,0)}, 중증(1~7급)={dist_d.get(1,0)}, "
      f"경증(8~14급)={dist_d.get(2,0)}")


장해등급: 무장해=521, 중증(1~7급)=290, 경증(8~14급)=1917


In [29]:
# ============================================================
# 경제활동상태 재코딩 (차수별)
#    emp0x2: 1=취업, 2=실업, 3=비경활
# ============================================================

for w in range(1, 6):
    col = f'emp0{w}2'
    if col in df.columns:
        df[f'empstat_w{w}'] = pd.to_numeric(df[col], errors='coerce')
        df.loc[~df[f'empstat_w{w}'].isin([1, 2, 3]), f'empstat_w{w}'] = np.nan

print("\n[차수별 경제활동상태]")
emp_label = {1: '취업', 2: '실업', 3: '비경활'}
for w in range(1, 6):
    col = f'empstat_w{w}'
    if col in df.columns:
        dist = df[col].value_counts().sort_index()
        print(f"  {w}차: " + ", ".join(
            [f"{emp_label.get(k,'?')}={v}" for k, v in dist.items()]))


[차수별 경제활동상태]
  1차: 취업=1698, 실업=1030
  2차: 취업=1833, 실업=821
  3차: 취업=1846, 실업=804
  4차: 취업=1910, 실업=783
  5차: 취업=1910, 실업=818


In [30]:
# ============================================================
# 심리·건강 변수 처리
#    자아존중감  G0x021001: 원척도 1(매우그렇다)~4(전혀그렇지않다) → 역코딩
#    생활만족도  G0x022007: 전반적 생활만족도
#    주관적건강  G0x005001: 1=매우나쁨 ~ 4=매우좋음
# ============================================================

for w in range(1, 6):
    ww = f'0{w}'
    # 자아존중감 역코딩 (높을수록 긍정)
    ec = f'G{ww}021001'
    if ec in df.columns:
        df[f'esteem_w{w}'] = 5 - pd.to_numeric(df[ec], errors='coerce')

    # 전반적 생활만족도
    sc = f'G{ww}022007'
    if sc in df.columns:
        df[f'lifesatis_w{w}'] = pd.to_numeric(df[sc], errors='coerce')

    # 주관적 건강상태
    hc = f'G{ww}005001'
    if hc in df.columns:
        df[f'health_w{w}'] = pd.to_numeric(df[hc], errors='coerce')

print("\n[5차 심리·건강 기초통계]")
for col, label in [('esteem_w5','자아존중감'), ('lifesatis_w5','생활만족도'), ('health_w5','주관적건강')]:
    if col in df.columns:
        s = df[col].dropna()
        print(f"  {label}: M={s.mean():.2f}, SD={s.std():.2f}, n={len(s)}")



[5차 심리·건강 기초통계]
  자아존중감: M=2.11, SD=0.70, n=2728
  생활만족도: M=2.62, SD=0.66, n=2728
  주관적건강: M=2.58, SD=0.67, n=2728


In [31]:
# ============================================================
# 재활서비스 이진화
#    jobservice05: 직업재활 (훈련/취업지원)
#    socservice05: 사회재활 (심리상담/취미)
# ============================================================

for col, new_col in [('jobservice05', 'job_rehab'), ('socservice05', 'soc_rehab')]:
    if col in df.columns:
        s = pd.to_numeric(df[col], errors='coerce')
        df[new_col] = np.where(s.isna(), np.nan, (s >= 1).astype(float))

print(f"\n직업재활 이용: {df['job_rehab'].sum():.0f}명 | "
      f"사회재활 이용: {df['soc_rehab'].sum():.0f}명")



직업재활 이용: 435명 | 사회재활 이용: 489명


In [32]:
# ============================================================
# 최종 분석 데이터셋 구성 (LCGA 입력용)
# ============================================================

final_cols = [
    'pid',
    # Y 변수: RIRI (3~5차)
    'RIRI_w3', 'RIRI_w4', 'RIRI_w5',
    'RIRI_class_w3', 'RIRI_class_w4', 'RIRI_class_w5',
    # 원천 소득 (참고용)
    'H03003036', 'H04003036', 'H05003036',
    'H03002005', 'H04002005', 'H05002005',
    'H03003001', 'H04003001', 'H05003001',
    # 인구사회학적
    'gender', 'age054', 'edu05', 'area056',
    # 장해
    'disa052', 'disa0515_num', 'disa_class',
    # 재활서비스
    'service05', 'job_rehab', 'soc_rehab',
    # 경제활동상태 시계열
    'empstat_w1', 'empstat_w2', 'empstat_w3', 'empstat_w4', 'empstat_w5',
    # 자아존중감 시계열
    'esteem_w1', 'esteem_w2', 'esteem_w3', 'esteem_w4', 'esteem_w5',
    # 생활만족도 시계열
    'lifesatis_w1', 'lifesatis_w2', 'lifesatis_w3', 'lifesatis_w4', 'lifesatis_w5',
    # 주관적 건강상태 시계열
    'health_w1', 'health_w2', 'health_w3', 'health_w4', 'health_w5',
]

exist_cols = [c for c in final_cols if c in df.columns]
df_final   = df[exist_cols].copy()

# LCGA 대상: RIRI 3~5차 중 2개 시점 이상 유효
riri_obs = df_final[['RIRI_w3', 'RIRI_w4', 'RIRI_w5']].notna().sum(axis=1)
df_lcga  = df_final[riri_obs >= 2].copy()

print(f"\n최종 데이터: {df_final.shape}")
print(f"LCGA 분석 대상 (RIRI 2시점 이상): {len(df_lcga)}명")

# 결측 요약
print("\n[주요 변수 결측 요약]")
for col in ['RIRI_w3','RIRI_w4','RIRI_w5','disa_class','esteem_w1','lifesatis_w1']:
    if col in df_lcga.columns:
        miss = df_lcga[col].isna().sum()
        print(f"  {col}: 유효={df_lcga[col].notna().sum()}, 결측={miss}({miss/len(df_lcga)*100:.1f}%)")



최종 데이터: (2728, 46)
LCGA 분석 대상 (RIRI 2시점 이상): 2542명

[주요 변수 결측 요약]
  RIRI_w3: 유효=2436, 결측=106(4.2%)
  RIRI_w4: 유효=2482, 결측=60(2.4%)
  RIRI_w5: 유효=2472, 결측=70(2.8%)
  disa_class: 유효=2542, 결측=0(0.0%)
  esteem_w1: 유효=2542, 결측=0(0.0%)
  lifesatis_w1: 유효=2542, 결측=0(0.0%)


전처리 단계별 요약
1. 표본 선정
- 전체 패널 3,294명 중 5차년도 실제 응답자(p05 == 1)만 필터링 → 2,728명
- 탈락자(사망, 거절, 주소불명 등) 제외

2. 소득 변수 정제

사용 변수 3종 × 5개 차수 = 15개 변수 처리

- H0x003036 개인소득 총계 (근로+비근로 모두 포함)
- H0x002005 근로소득 총계
- H0x003001 휴업급여 수령액
    - 응답거절(9999998), 모름(9999999) → NaN으로 치환
    - 단위 변환: 만원/연 → 만원/월 (÷12), _m 접미사로 저장

3. **RIRI** 계산

$RIRI_w = \frac{Q_{irr} * W_{pre}}{LW_w}*\frac{CPI_{pre}}{CPI_w}*100$

- Q_irr = H0x002005_m / H0x003036_m → 전체 소득 중 근로소득이 차지하는 비중 (0~1 클리핑)
- W_pre = H03003036_m → 3차년도 월 소득을 기준선(baseline)으로 고정
- LW_w = 해당 차수의 서울시 생활임금 (외부 상수)
- CPI 보정 = 물가 변동 반영 (3차 기준 → 4·5차 실질 비교)
- 상위 1% 윈저라이징 → 극단값 처리
- 3차·4차·5차 각각 계산 → RIRI_w3, RIRI_w4, RIRI_w5


4. RIRI 3분류 레이블 생성
- RIRI >= 100, 완전회복, 2
- 50 <= RIRI < 100, 부분회복, 1
- RIRI < 50, 미회복, 0

→ RIRI_class_w3/4/5 변수로 저장 (LCGA 보조 해석용)

5. 장해등급 3분류
- disa056, 6범주 (1~3급/4~7급/8~9급/10~12급/13~14급/무장해) -> disa_class_6
- disa0515, 15범주 (1~14급 + 15=무장해) -> disa_class (주 사용)

-> 0=무장해, 1=중증(1~7급), 2=경증(8~14급) 로 통일

6. 경제활동상태 재코딩
- emp0x31=취업자, 2=실업자, 3=비경활
    - 주 분석용 → empstat3_w1~5
- emp0x61=원직장복귀, 2=재취업, 3=자영업, 4=무급가족, 5=실업, 6=비경활
    - 복귀 유형 파악용 → empstat6_w1~5

7. 심리·건강 변수 처리
- G0x021001 자아존중감1=그렇지않다 ~ 4=항상그렇다
    - 그대로 (높을수록 긍정)
- G0x022007 생활만족도1=매우만족 ~ 5=매우불만족
    - 역코딩 (6-x) → 높을수록 만족
- G0x005001 주관적건강1=매우나쁨 ~ 4=매우좋음
    - 그대로 (높을수록 좋음)
- G0x023001~006 자기효능감1=전혀그렇지않다 ~ 5=매우그렇다
    - 6문항 평균 → efficacy_w1~5

8. 재활서비스 이진화
- 코드북: 1=이용, 2=미이용 (0이 미이용이 아님)
- 수정 후: 1→1, 2→0, 그 외→NaN 으로 변환
- any_rehab (전체), job_rehab (직업재활), soc_rehab (사회재활)